<div class='alert alert-warning'>

# JupyterLite warning

- Running the **scikit-plots 0.5.dev0** interactive examples in JupyterLite is experimental and may not always work as expected.
- With high load times especially on low-resource platforms, and the version of scikit-plots might not be in sync with the one you are browsing the documentation for.
- If you encounter any issues, please report them on the [scikit-plots issue tracker](https://github.com/scikit-plots/scikit-plots/issues/new/choose).
- `micropip/piplite/pip` use `%pip` in JupyterLite instead of `pip` or `!pip`

```python
## Installing the dependencies first, and then scikit-plots from Anaconda.org.
import piplite; await piplite.install(  # or micropip
   'scikit-plots==0.5.dev0',            # Download scikit-plots *pyodide_20XX_0_wasm32.whl
   index_urls='https://pypi.anaconda.org/scikit-plots-wheels-staging-nightly/simple',
); import sklearn; import scikitplot as sp; sp.show_versions();
 ```

</div>

Single file, no embedding:


In [ ]:
from pathlib import Path
from scikitplot.corpus import CorpusPipeline, ParagraphChunker
pipeline = CorpusPipeline(chunker=ParagraphChunker())
result = pipeline.run(Path("article.txt"))
print(f"{result.n_documents} chunks from {result.source}")

Batch processing with sentence chunking:


In [ ]:
from scikitplot.corpus import CorpusPipeline, SentenceChunker, ExportFormat
pipeline = CorpusPipeline(
    # chunker=SentenceChunker(SentenceChunkerConfig(backend=SentenceBackend.NLTK)),
    chunker=SentenceChunker("en_core_web_sm"),  # default backend spacy
    output_path=Path("output/"),
    export_format=ExportFormat.PARQUET,
)
results = pipeline.run_batch(list(Path("corpus/").glob("*.txt")))

URL ingestion:


In [ ]:
# https://archive.org/download/WHO-documents
# https://www.who.int/europe/news/item/...
result = pipeline.run_url("https://en.wikipedia.org/wiki/Python")

YouTube transcript:


In [ ]:
result = pipeline.run("https://www.youtube.com/watch?v=rwPISgZcYIk")

Image OCR:


In [ ]:
reader = DocumentReader.create(Path("scan.png"))
docs = list(reader.get_documents())

Video transcription (subtitle-first):


In [ ]:
# Richard Feynman - The Character of Physical Law (1964) - Complete - Better Audio
# https://www.youtube.com/watch?v=kEx-gRfuhhk
reader = DocumentReader.create(Path("lecture.mp4"))
docs = list(reader.get_documents())

With embeddings:


In [ ]:
from scikitplot.corpus import EmbeddingEngine
engine = EmbeddingEngine(backend="sentence_transformers")
pipeline = CorpusPipeline(
    chunker=ParagraphChunker(),
    embedding_engine=engine,
)
result = pipeline.run(Path("article.txt"))
result.documents[0].has_embedding

True

Convenience function (direct replacement for remarx ``create_corpus``):


In [ ]:
from scikitplot.corpus import create_corpus
result = create_corpus(
    input_path=Path("chapter01.txt"),
    output_path=Path("output/chapter01.csv"),
)

In [ ]:
from scikitplot.corpus import CorpusBuilder, BuilderConfig
builder = CorpusBuilder(
    BuilderConfig(
        chunker="paragraph",
        normalize=True,
        enrich=True,
        embed=True,
        build_index=True,
    )
)
result = builder.build("./data/")
results = builder.search("quantum computing")
lc_docs = builder.to_langchain()
mcp_response = builder.to_mcp_tool_result("quantum computing")